In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Mode: 'sector' or 'tenor' ---
MODE = 'sector'  # 'sector' or 'tenor'

SECTORS = {
    'Belly': ['6Y', '7Y', '8Y'],
    '10Y Sector': ['9Y', '10Y', '11Y'],
    'Long End': ['12Y', '13Y', '14Y', '15Y'],
}

SELECTED = 'Belly'  # sector name if MODE='sector', or tenor string like '10Y' if MODE='tenor'

COUNTRIES = {
    'Peru': df_perugb_cmt,
    'Mexico': df_mbono_cmt,
    'Colombia': df_coltes_cmt,
    'Chile': df_btpcl_cmt,
}

FOCUS_COUNTRY = 'Peru'
LOOKBACK_START = '2022-01-01'  # controls everything — no data before this date used anywhere
N_PCS = 2  # None = auto via 90% cumvar

SD_WINDOW = 252
SD_BANDS = [1.25, 1.65]
ROLLING_WINDOWS = [5, 10, 20]
ROLL_DISPLAY = [5, 20]
TRAIL_WINDOW = 60
ATTRIB_WINDOWS = [5, 10, 20]

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.15,
    'grid.linestyle': '--',
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 120,
})

# institutional palette
COLORS = {
    'pc1': '#4A6FA5',      # steel blue
    'pc2': '#C47B3B',      # copper
    'pc3': '#6B8F6B',      # sage
    'mean': '#8B8B8B',     # warm gray
    'actual': '#1A1A1A',   # near black
    'pos_inner': '#FFDAB9', # light peach
    'pos_outer': '#CD5C5C', # indian red
    'neg_inner': '#B0D4F1', # light steel blue
    'neg_outer': '#2E5E8E', # dark steel blue
    'bull': '#C8E6C9',     # light green
    'bear': '#FFCDD2',     # light red
}

In [ ]:
def build_spread_panel(countries, mode, selected, sectors, ust_df, lookback_start):
    # determine UST series
    ust = ust_df.set_index('Fecha') if 'Fecha' in ust_df.columns else ust_df.copy()
    if mode == 'sector':
        cols = sectors[selected]
        ust_series = ust[cols].mean(axis=1)
    else:
        ust_series = ust[selected]
    result = {}
    for name, df in countries.items():
        cty = df.set_index('Fecha') if 'Fecha' in df.columns else df.copy()
        if mode == 'sector':
            cty_series = cty[cols].mean(axis=1)
        else:
            cty_series = cty[selected]
        # intersect dates
        idx = cty_series.index.intersection(ust_series.index)
        spread = (cty_series.loc[idx] - ust_series.loc[idx]) * 100
        result[name] = spread
    panel = pd.DataFrame(result)
    panel.index = pd.to_datetime(panel.index)
    # filter to lookback_start immediately
    panel = panel.loc[panel.index >= pd.Timestamp(lookback_start)]
    return panel

df_spreads = build_spread_panel(COUNTRIES, MODE, SELECTED, SECTORS, df_ust_cmt, LOOKBACK_START)
print(f'Mode: {MODE} | Selected: {SELECTED}')
print(f'Shape: {df_spreads.shape} | {df_spreads.index[0].date()} to {df_spreads.index[-1].date()}')
print(df_spreads.head(3))

In [ ]:
def run_pca(df_spreads, n_pcs):
    data = df_spreads.dropna()
    means = data.mean()
    demeaned = data - means
    cov = demeaned.cov().values
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    # sort descending
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    total_var = eigenvalues.sum()
    var_explained = eigenvalues / total_var
    cum_var = np.cumsum(var_explained)
    # determine n_pcs
    if n_pcs is None:
        k = int(np.searchsorted(cum_var, 0.90)) + 1
    else:
        k = n_pcs
    loadings = pd.DataFrame(
        eigenvectors[:, :k],
        index=data.columns,
        columns=[f'PC{i+1}' for i in range(k)]
    )
    scores = pd.DataFrame(
        demeaned.values @ eigenvectors[:, :k],
        index=data.index,
        columns=[f'PC{i+1}' for i in range(k)]
    )
    return {
        'means': means,
        'loadings': loadings,
        'scores': scores,
        'eigenvalues': eigenvalues[:k],
        'var_explained': var_explained[:k],
        'cum_var_explained': cum_var[:k],
        'n_pcs': k,
    }

def decompose(pca, df_spreads):
    data = df_spreads.dropna()
    loadings = pca['loadings']
    means = pca['means']
    demeaned = data - means
    scores_full = pd.DataFrame(
        demeaned.values @ loadings.values,
        index=data.index,
        columns=loadings.columns
    )
    fitted = pd.DataFrame(
        scores_full.values @ loadings.values.T + means.values,
        index=data.index,
        columns=data.columns
    )
    residuals = data - fitted
    # per-factor contributions per country: score_k * loading_country_k
    contributions = {}
    for country in data.columns:
        contribs = pd.DataFrame(
            {pc: scores_full[pc] * loadings.loc[country, pc] for pc in loadings.columns},
            index=data.index
        )
        contributions[country] = contribs
    return {
        'fitted': fitted,
        'residuals': residuals,
        'contributions': contributions,
        'scores_full': scores_full,
    }

pca = run_pca(df_spreads, N_PCS)
decomp = decompose(pca, df_spreads)
print(f'n_pcs: {pca["n_pcs"]}')
print(f'Var explained: {[f"{v:.1%}" for v in pca["var_explained"]]}')
print(f'Cumulative: {[f"{v:.1%}" for v in pca["cum_var_explained"]]}')
print(f'Score date range: {decomp["scores_full"].index[0].date()} to {decomp["scores_full"].index[-1].date()}')
print(f'Residual shape: {decomp["residuals"].shape}')

In [ ]:
# daily factor returns (first difference of scores)
factor_returns = decomp['scores_full'].diff().dropna()

# rolling sums for each window
roll_factor = {}
for w in ROLLING_WINDOWS:
    roll_factor[w] = factor_returns.rolling(w).sum()

print(f'Factor returns shape: {factor_returns.shape}')
print(f'Rolling windows computed: {list(roll_factor.keys())}')
print(factor_returns.describe().round(3))

In [ ]:
# --- Variance Explained Table ---
ve_df = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(pca['n_pcs'])],
    'Var Explained (%)': [f'{v:.2%}' for v in pca['var_explained']],
    'Cumulative (%)': [f'{v:.2%}' for v in pca['cum_var_explained']],
})
print(ve_df.to_string(index=False))

pc_colors = [COLORS['pc1'], COLORS['pc2'], COLORS['pc3']]
pcs = [f'PC{i+1}' for i in range(pca['n_pcs'])]
n = pca['n_pcs']

# --- Bar chart 1: Variance Explained per PC ---
fig, ax = plt.subplots(figsize=(max(4, n * 1.2), 4))
bars = ax.bar(pcs, pca['var_explained'] * 100, color=pc_colors[:n], width=0.5, edgecolor='white')
for bar, v in zip(bars, pca['var_explained']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{v:.1%}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title(f'Variance Explained per PC ({MODE.title()}: {SELECTED})')
ax.set_ylabel('Variance Explained (%)')
ax.set_ylim(0, max(pca['var_explained']) * 100 * 1.2)
plt.tight_layout()
plt.show()

# --- Bar chart 2: Grouped Loadings by Country ---
loadings_df = pca['loadings']
countries_list = list(loadings_df.index)
n_countries = len(countries_list)
x = np.arange(n_countries)
width = 0.8 / n

fig, ax = plt.subplots(figsize=(max(6, n_countries * 1.5), 4))
for i, pc in enumerate(pcs):
    offset = (i - (n - 1) / 2) * width
    vals = loadings_df[pc].values
    bars = ax.bar(x + offset, vals, width=width * 0.9,
                  label=pc, color=pc_colors[i], edgecolor='white')
    for bar, v in zip(bars, vals):
        va = 'bottom' if v >= 0 else 'top'
        y_off = 0.005 if v >= 0 else -0.005
        ax.text(bar.get_x() + bar.get_width() / 2, v + y_off,
                f'{v:.2f}', ha='center', va=va, fontsize=7)
ax.axhline(0, color='#555555', linewidth=0.8, linestyle='-')
ax.set_xticks(x)
ax.set_xticklabels(countries_list)
ax.set_title(f'PCA Loadings by Country ({"Sector" if MODE == "sector" else "Tenor"}: {SELECTED})')
ax.set_ylabel('Loading')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Part 1: Loading vs Volatility ---
spread_changes = df_spreads.dropna().diff().dropna()
sd_per_country = spread_changes.std()

# SD ratios relative to first country
base = sd_per_country.iloc[0]
sd_ratios = sd_per_country / base

pc1_loadings = pca['loadings']['PC1'].abs()
base_load = pc1_loadings.iloc[0]
load_ratios = pc1_loadings / base_load

ratio_df = pd.DataFrame({
    'SD (bps)': sd_per_country.round(2),
    'SD Ratio': sd_ratios.round(3),
    '|PC1 Loading|': pc1_loadings.round(4),
    'Loading Ratio': load_ratios.round(3),
    'Ratio Match': (sd_ratios / load_ratios).round(3),
})
print('Loading vs Volatility (base = first country):')
print(ratio_df.to_string())

# --- Part 2: PCA on Correlation Matrix ---
def run_pca_corr(df_spreads, n_pcs):
    data = df_spreads.dropna()
    stds = data.std()
    standardized = data / stds
    corr = standardized.cov().values
    eigenvalues, eigenvectors = np.linalg.eigh(corr)
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    total_var = eigenvalues.sum()
    var_explained = eigenvalues / total_var
    cum_var = np.cumsum(var_explained)
    if n_pcs is None:
        k = int(np.searchsorted(cum_var, 0.90)) + 1
    else:
        k = n_pcs
    loadings = pd.DataFrame(
        eigenvectors[:, :k],
        index=data.columns,
        columns=[f'PC{i+1}' for i in range(k)]
    )
    return {
        'loadings': loadings,
        'var_explained': var_explained[:k],
        'cum_var_explained': cum_var[:k],
        'n_pcs': k,
    }

pca_corr = run_pca_corr(df_spreads, N_PCS)

# print side-by-side
cov_loads = pca['loadings'].copy()
corr_loads = pca_corr['loadings'].copy()
cov_loads.columns = [f'Cov_{c}' for c in cov_loads.columns]
corr_loads.columns = [f'Corr_{c}' for c in corr_loads.columns]
combined = pd.concat([cov_loads, corr_loads], axis=1)
print('\nCovariance vs Correlation Loadings:')
print(combined.round(4).to_string())

# side-by-side bar charts
pcs = [f'PC{i+1}' for i in range(pca['n_pcs'])]
n = len(pcs)
pc_colors = [COLORS['pc1'], COLORS['pc2'], COLORS['pc3']]
countries_list = list(pca['loadings'].index)
n_countries = len(countries_list)
x = np.arange(n_countries)
width = 0.8 / n

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'Covariance vs Correlation Loadings ({MODE.title()}: {SELECTED})',
             fontsize=12, fontweight='bold')

for ax_i, (ax, lds, label) in enumerate([
    (axes[0], pca['loadings'], 'Covariance PCA'),
    (axes[1], pca_corr['loadings'], 'Correlation PCA'),
]):
    for i, pc in enumerate(pcs):
        offset = (i - (n - 1) / 2) * width
        vals = lds[pc].values
        bars = ax.bar(x + offset, vals, width=width * 0.9,
                      label=pc, color=pc_colors[i], edgecolor='white')
        for bar, v in zip(bars, vals):
            va = 'bottom' if v >= 0 else 'top'
            y_off = 0.005 if v >= 0 else -0.005
            ax.text(bar.get_x() + bar.get_width() / 2, v + y_off,
                    f'{v:.2f}', ha='center', va=va, fontsize=7)
    ax.axhline(0, color='#555555', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(countries_list)
    ax.set_title(label)
    ax.set_ylabel('Loading')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Cell 7: PC Score Levels with Regime Shading ---
scores = decomp['scores_full']
n_pcs = pca['n_pcs']
pc_line_colors = [COLORS['pc1'], COLORS['pc2'], COLORS['pc3']]

fig, axes = plt.subplots(n_pcs, 1, figsize=(12, 3 * n_pcs), sharex=True)
if n_pcs == 1:
    axes = [axes]

for i, pc in enumerate([f'PC{k+1}' for k in range(n_pcs)]):
    ax = axes[i]
    s = scores[pc]
    trail = s.rolling(TRAIL_WINDOW).mean()
    color = pc_line_colors[i]
    # regime shading
    ax.fill_between(s.index, s.min() * 1.5, s.max() * 1.5,
                    where=(s >= trail), color=COLORS['bull'], alpha=0.5, label='_nolegend_')
    ax.fill_between(s.index, s.min() * 1.5, s.max() * 1.5,
                    where=(s < trail), color=COLORS['bear'], alpha=0.5, label='_nolegend_')
    ax.plot(s.index, s, color=color, linewidth=1.2, label=f'{pc} level')
    ax.plot(trail.index, trail, color=color, linewidth=1.0, linestyle='--',
            alpha=0.55, label=f'{TRAIL_WINDOW}d mean')
    ax.axhline(0, color='#999999', linewidth=0.7)
    ax.set_ylabel('Score (bps)')
    ax.legend(loc='upper left')
    ax.set_xlim(scores.index[0], scores.index[-1])
    # keep y-axis tight to actual data range
    pad = (s.max() - s.min()) * 0.05
    ax.set_ylim(s.min() - pad, s.max() + pad)
    ax.set_title(pc)

fig.suptitle(f'PC Scores — Levels & Regime ({MODE}: {SELECTED})', y=1.01)
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
# --- Cell 8: Factor Return Momentum ---
roll_colors = [COLORS['pc1'], COLORS['pc2'], COLORS['mean'], COLORS['pc3']]
n_pcs = pca['n_pcs']

fig, axes = plt.subplots(n_pcs, 1, figsize=(12, 3 * n_pcs), sharex=True)
if n_pcs == 1:
    axes = [axes]

for i, pc in enumerate([f'PC{k+1}' for k in range(n_pcs)]):
    ax = axes[i]
    for j, w in enumerate(ROLL_DISPLAY):
        series = roll_factor[w][pc].dropna()
        ax.plot(series.index, series, color=roll_colors[j], linewidth=1.2, label=f'{w}d sum')
    ax.axhline(0, color='#999999', linewidth=0.7)
    ax.set_ylabel('Rolling Sum (bps)')
    ax.legend(loc='upper left')
    ax.set_xlim(factor_returns.index[0], factor_returns.index[-1])
    ax.set_title(pc)

fig.suptitle(f'Factor Return Momentum ({MODE}: {SELECTED})', y=1.01)
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
# --- Cell 9: Regime Summary Table (Client-Facing) ---
def regime_summary(scores_full, roll_factor, pca, trail_window):
    latest = scores_full.index[-1]
    rows = []
    for i in range(pca['n_pcs']):
        pc = f'PC{i+1}'
        s = scores_full[pc]
        level = float(s.iloc[-1])
        pctile = float((s < level).mean() * 100)
        mom5 = float(roll_factor[5][pc].iloc[-1])
        mom20 = float(roll_factor[20][pc].iloc[-1])
        trail_mean = float(s.rolling(trail_window).mean().iloc[-1])
        regime = 'Above trend' if level > trail_mean else 'Below trend'
        # direction: PC1 positive score = spreads wide (loadings typically positive)
        # use 20d momentum sign to determine direction
        if pc == 'PC1':
            direction = 'Compressing' if mom20 < 0 else 'Widening'
        else:
            direction = 'Differentiating'
        rows.append({
            'Factor': pc,
            'Level': level,
            '%ile': pctile,
            '5d Mom': mom5,
            '20d Mom': mom20,
            'Direction': direction,
            'Regime': regime,
        })
    # PC2 differentiation intensity
    pc2_level = float(scores_full['PC2'].iloc[-1]) if pca['n_pcs'] >= 2 else None
    pc2_pctile = float((scores_full['PC2'] < pc2_level).mean() * 100) if pc2_level is not None else None
    diff_intensity = abs(pc2_pctile - 50) if pc2_pctile is not None else None
    return rows, latest, diff_intensity, pc2_pctile

rows, latest, diff_intensity, pc2_pctile = regime_summary(
    decomp['scores_full'], roll_factor, pca, TRAIL_WINDOW
)

# format and print
header = f'=== LatAm LC Regime Monitor | {MODE}: {SELECTED} | {latest.date()} ==='
print(header)
print()
col_w = [8, 8, 6, 9, 9, 18, 12]
hdrs = ['Factor', 'Level', '%ile', '5d Mom', '20d Mom', 'Direction', 'Regime']
print(''.join(h.ljust(col_w[j]) for j, h in enumerate(hdrs)))
for r in rows:
    vals = [
        r['Factor'],
        f"{r['Level']:+.1f}",
        f"{r['%ile']:.0f}%",
        f"{r['5d Mom']:+.1f}",
        f"{r['20d Mom']:+.1f}",
        r['Direction'],
        r['Regime'],
    ]
    print(''.join(str(v).ljust(col_w[j]) for j, v in enumerate(vals)))

if diff_intensity is not None:
    if diff_intensity >= 35:
        intensity_label = 'High'
    elif diff_intensity >= 15:
        intensity_label = 'Moderate'
    else:
        intensity_label = 'Low'
    print(f'\nDifferentiation intensity: {intensity_label} (PC2 at {pc2_pctile:.0f}th percentile)')

In [ ]:
# --- Cell 10: Client Flow Anticipation ---
def classify_momentum_state(fast, slow, ratio=5/20):
    # returns string label for each date
    states = []
    for f, s in zip(fast, slow):
        if pd.isna(f) or pd.isna(s):
            states.append('unknown')
            continue
        same_sign = (f > 0 and s > 0) or (f < 0 and s < 0)
        building = abs(f) > abs(s) * ratio
        if f > 0 and s > 0 and building:
            states.append('accel_compress')
        elif f < 0 and s < 0 and building:
            states.append('accel_widen')
        elif not same_sign:
            states.append('transition')
        else:
            states.append('decel')
    return states

fast = roll_factor[5]['PC1']
slow = roll_factor[20]['PC1']
accel = fast.abs() - slow.abs() * (5 / 20)

states = classify_momentum_state(fast.values, slow.values)
state_series = pd.Series(states, index=fast.index)

state_color_map = {
    'accel_compress': COLORS['bull'],
    'accel_widen': COLORS['bear'],
    'decel': '#E0E0E0',
    'transition': '#E0E0E0',
    'unknown': '#FFFFFF',
}

# bar colors for acceleration panel
bar_colors = []
for st, av in zip(state_series, accel):
    if st == 'accel_compress':
        bar_colors.append(COLORS['pc3'])   # sage green
    elif st == 'accel_widen':
        bar_colors.append(COLORS['pos_outer'])  # indian red
    else:
        bar_colors.append(COLORS['mean'])   # gray

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
fig.suptitle(f'Client Flow Anticipation — PC1 ({MODE}: {SELECTED})', y=1.01)

# top: fast and slow rolling sums with regime shading
valid = fast.dropna().index
ax1.fill_between(state_series.index,
                 fast.min() * 1.5, fast.max() * 1.5,
                 where=(state_series == 'accel_compress'),
                 color=COLORS['bull'], alpha=0.4, label='_nolegend_')
ax1.fill_between(state_series.index,
                 fast.min() * 1.5, fast.max() * 1.5,
                 where=(state_series == 'accel_widen'),
                 color=COLORS['bear'], alpha=0.4, label='_nolegend_')
ax1.plot(fast.index, fast, color=COLORS['pc1'], linewidth=1.2, label='5d sum')
ax1.plot(slow.index, slow, color=COLORS['pc2'], linewidth=1.2,
         linestyle='--', label='20d sum')
ax1.axhline(0, color='#999999', linewidth=0.7)
ax1.set_ylabel('Rolling Sum (bps)')
ax1.legend(loc='upper left')
fmin, fmax = fast.min(), fast.max()
pad1 = (fmax - fmin) * 0.05
ax1.set_ylim(fmin - pad1, fmax + pad1)

# bottom: acceleration bars
accel_clean = accel.dropna()
state_clean = state_series.loc[accel_clean.index]
colors_clean = []
for st, av in zip(state_clean, accel_clean):
    if st == 'accel_compress':
        colors_clean.append(COLORS['pc3'])
    elif st == 'accel_widen':
        colors_clean.append(COLORS['pos_outer'])
    else:
        colors_clean.append(COLORS['mean'])

ax2.bar(accel_clean.index, accel_clean.values, color=colors_clean,
        width=1.5, linewidth=0)
ax2.axhline(0, color='#999999', linewidth=0.7)
ax2.set_ylabel('Acceleration (bps)')
ax2.set_xlabel('Date')

# legend patches
from matplotlib.patches import Patch
legend_els = [
    Patch(facecolor=COLORS['pc3'], label='Accel compression'),
    Patch(facecolor=COLORS['pos_outer'], label='Accel widening'),
    Patch(facecolor=COLORS['mean'], label='Decel / Transition'),
]
ax2.legend(handles=legend_els, loc='upper left')
ax1.set_xlim(fast.dropna().index[0], fast.index[-1])

plt.tight_layout()
plt.show()

# --- Print current signal ---
interpretations = {
    'accel_compress': 'Regional compression trend is gaining momentum. Expect continued bid for LatAm LC. Flow likely skewed to receivers / duration longs.',
    'accel_widen': 'Regional widening trend is accelerating. Selling pressure building across the bloc. Watch for stop-loss flows amplifying moves.',
    'decel': 'Momentum trend is losing pace. Conviction is fading; watch for reversal or consolidation before next directional move.',
    'transition': 'Short and medium-term signals are conflicting. Market is at an inflection point. Positioning may be caught offsides.',
    'unknown': 'Insufficient data to classify current momentum state.',
}

fast_val = float(fast.iloc[-1]) if not fast.empty else float('nan')
slow_val = float(slow.iloc[-1]) if not slow.empty else float('nan')
cur_state = state_series.iloc[-1] if not state_series.empty else 'unknown'
cur_date = fast.index[-1].date()

fast_dir = 'compression' if fast_val < 0 else 'widening'
slow_dir = 'compression' if slow_val < 0 else 'widening'

state_labels = {
    'accel_compress': 'Accelerating compression',
    'accel_widen': 'Accelerating widening',
    'decel': 'Decelerating (pace is slowing)',
    'transition': 'Transition (signals conflicting)',
    'unknown': 'Unknown',
}

print(f'=== Client Flow Anticipation | {MODE}: {SELECTED} | {cur_date} ===')
print()
print(f'PC1 5d momentum:  {fast_val:+.1f} bps ({fast_dir})')
print(f'PC1 20d momentum: {slow_val:+.1f} bps ({slow_dir})')
print(f'Momentum state:   {state_labels.get(cur_state, cur_state)}')
print()
print(f'Interpretation: {interpretations.get(cur_state, "")}' )


In [ ]:
# --- Cell 11: Main Diagnostic Chart (FOCUS_COUNTRY) ---
def plot_diagnostic(country, decomp, df_spreads, pca, sd_window, sd_bands, mode, selected, lookback_start):
    actual = df_spreads[country].dropna()
    fitted = decomp['fitted'][country].reindex(actual.index).dropna()
    residual = decomp['residuals'][country].reindex(actual.index).dropna()
    contribs = decomp['contributions'][country].reindex(actual.index).dropna()
    means_val = pca['means'][country]
    pc_cols = pca['loadings'].columns.tolist()
    pc_color_map = {'PC1': COLORS['pc1'], 'PC2': COLORS['pc2'], 'PC3': COLORS['pc3']}

    roll_sd = residual.rolling(sd_window).std()

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

    # --- top panel: actual vs stacked contributions ---
    # stack: mean base + PC contributions
    base = pd.Series(means_val, index=actual.index)
    cumstack = base.copy()
    ax1.fill_between(base.index, 0, base, color=COLORS['mean'], alpha=0.6, label='Mean')
    for pc in pc_cols:
        top = cumstack + contribs[pc]
        ax1.fill_between(contribs.index, cumstack, top,
                         color=pc_color_map.get(pc, COLORS['mean']),
                         alpha=0.6, label=pc)
        cumstack = top
    ax1.plot(actual.index, actual, color=COLORS['actual'], linewidth=2, label='Actual', zorder=5)
    ax1.set_ylabel('Spread (bps)')
    ax1.legend(loc='upper left', ncol=len(pc_cols) + 2)
    ax1.set_xlim(actual.index[0], actual.index[-1])

    # --- bottom panel: residual with SD bands ---
    inner_sd = sd_bands[0] * roll_sd
    outer_sd = sd_bands[1] * roll_sd

    # positive dislocation shading
    ax2.fill_between(residual.index, inner_sd, outer_sd,
                     where=(residual >= inner_sd),
                     color=COLORS['pos_inner'], alpha=0.7, label='_nolegend_')
    ax2.fill_between(residual.index, outer_sd, residual,
                     where=(residual >= outer_sd),
                     color=COLORS['pos_outer'], alpha=0.3, label='_nolegend_')
    # negative dislocation shading
    ax2.fill_between(residual.index, -outer_sd, -inner_sd,
                     where=(residual <= -inner_sd),
                     color=COLORS['neg_inner'], alpha=0.7, label='_nolegend_')
    ax2.fill_between(residual.index, residual, -outer_sd,
                     where=(residual <= -outer_sd),
                     color=COLORS['neg_outer'], alpha=0.3, label='_nolegend_')

    # SD band lines
    for thresh, sd_series in zip(sd_bands, [inner_sd, outer_sd]):
        ax2.plot(sd_series.index, sd_series, color='#AAAAAA', linewidth=0.7,
                 linestyle='--', label=f'+{thresh}σ')
        ax2.plot(sd_series.index, -sd_series, color='#AAAAAA', linewidth=0.7,
                 linestyle='--', label=f'-{thresh}σ')

    ax2.plot(residual.index, residual, color=COLORS['actual'], linewidth=1.2, label='Residual', zorder=5)
    ax2.axhline(0, color='#999999', linewidth=0.7)
    ax2.set_ylabel('Residual (bps)')
    ax2.set_xlabel('Date')
    ax2.legend(loc='upper left', ncol=3)
    ax2.set_xlim(actual.index[0], actual.index[-1])

    fig.suptitle(
        f'{country} | {mode}: {selected} | Spread vs UST | PCA from {lookback_start}',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

plot_diagnostic(FOCUS_COUNTRY, decomp, df_spreads, pca,
                SD_WINDOW, SD_BANDS, MODE, SELECTED, LOOKBACK_START)


In [ ]:
# --- Cell 12: All Countries Residual Dashboard ---
def plot_residual_dashboard(decomp, df_spreads, pca, sd_window, sd_bands, mode, selected):
    countries = list(df_spreads.columns)
    n = len(countries)
    ncols = 2
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4 * nrows), sharex=False)
    axes_flat = axes.flatten() if n > 1 else [axes]

    for idx, country in enumerate(countries):
        ax = axes_flat[idx]
        residual = decomp['residuals'][country].dropna()
        roll_sd = residual.rolling(sd_window).std()
        inner_sd = sd_bands[0] * roll_sd
        outer_sd = sd_bands[1] * roll_sd

        ax.fill_between(residual.index, inner_sd, outer_sd,
                         where=(residual >= inner_sd),
                         color=COLORS['pos_inner'], alpha=0.7)
        ax.fill_between(residual.index, outer_sd, residual,
                         where=(residual >= outer_sd),
                         color=COLORS['pos_outer'], alpha=0.3)
        ax.fill_between(residual.index, -outer_sd, -inner_sd,
                         where=(residual <= -inner_sd),
                         color=COLORS['neg_inner'], alpha=0.7)
        ax.fill_between(residual.index, residual, -outer_sd,
                         where=(residual <= -outer_sd),
                         color=COLORS['neg_outer'], alpha=0.3)

        for thresh, sd_s in zip(sd_bands, [inner_sd, outer_sd]):
            ax.plot(sd_s.index, sd_s, color='#AAAAAA', linewidth=0.7, linestyle='--')
            ax.plot(sd_s.index, -sd_s, color='#AAAAAA', linewidth=0.7, linestyle='--')

        ax.plot(residual.index, residual, color=COLORS['actual'], linewidth=1.0)
        ax.axhline(0, color='#999999', linewidth=0.7)
        ax.set_title(country)
        ax.set_ylabel('Residual (bps)')
        ax.set_xlim(residual.index[0], residual.index[-1])

    # hide unused subplots
    for j in range(n, len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle(f'All Countries — {mode}: {selected} | Spread Residuals vs UST',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_residual_dashboard(decomp, df_spreads, pca, SD_WINDOW, SD_BANDS, MODE, SELECTED)


In [ ]:
# --- Cell 13: Period Attribution Table (Client-Facing) ---
def period_attribution(decomp, df_spreads, country, windows):
    actual = df_spreads[country].dropna()
    fitted = decomp['fitted'][country].reindex(actual.index).dropna()
    residual = decomp['residuals'][country].reindex(actual.index).dropna()
    contribs = decomp['contributions'][country].reindex(actual.index).dropna()
    pc_cols = contribs.columns.tolist()
    rows = []
    for w in windows:
        actual_chg = float(actual.iloc[-1] - actual.iloc[-(w + 1)])
        pc_chgs = {pc: float(contribs[pc].iloc[-1] - contribs[pc].iloc[-(w + 1)]) for pc in pc_cols}
        resid_chg = float(residual.iloc[-1] - residual.iloc[-(w + 1)])
        resid_pct = abs(resid_chg) / abs(actual_chg) * 100 if actual_chg != 0 else 0.0
        row = {'Window': f'{w}d', 'Actual Chg': actual_chg}
        row.update(pc_chgs)
        row['Residual'] = resid_chg
        row['Residual %'] = resid_pct
        rows.append(row)
    return pd.DataFrame(rows).set_index('Window')

attrib = period_attribution(decomp, df_spreads, FOCUS_COUNTRY, ATTRIB_WINDOWS)
pc_cols = decomp['contributions'][FOCUS_COUNTRY].columns.tolist()
pc_labels = {'PC1': 'PC1 (Regional)', 'PC2': 'PC2 (Differentiation)'}

latest = decomp['scores_full'].index[-1].date()
print(f'=== Spread Attribution: {FOCUS_COUNTRY} | {MODE}: {SELECTED} | as of {latest} ===')
print()

# header
pc_hdr_parts = [pc_labels.get(pc, pc) for pc in pc_cols]
hdr = f'{"Window":>7}  {"Actual Chg":>11}  ' + '  '.join(f'{h:>22}' for h in pc_hdr_parts)
hdr += f'  {"Residual":>10}  {"Residual %":>10}'
print(hdr)

for win, row in attrib.iterrows():
    pc_parts = '  '.join(f'{row[pc]:>+21.1f} bps' for pc in pc_cols)
    line = (f'{win:>7}  {row["Actual Chg"]:>+10.1f} bps  {pc_parts}'
            f'  {row["Residual"]:>+9.1f} bps  {row["Residual %"]:>9.0f}%')
    print(line)


In [ ]:
# --- Cell 14: Current Snapshot Table (Client-Facing) ---
def current_snapshot(decomp, df_spreads, pca, sd_window):
    countries = list(df_spreads.columns)
    pc_cols = pca['loadings'].columns.tolist()
    rows = []
    for country in countries:
        actual = df_spreads[country].dropna()
        fitted = decomp['fitted'][country].reindex(actual.index).dropna()
        residual = decomp['residuals'][country].reindex(actual.index).dropna()
        contribs = decomp['contributions'][country].reindex(actual.index).dropna()
        roll_sd = residual.rolling(sd_window).std()
        zscore = float(residual.iloc[-1] / roll_sd.iloc[-1]) if roll_sd.iloc[-1] > 0 else 0.0
        row = {
            'Country': country,
            'Actual': float(actual.iloc[-1]),
            'Fitted': float(fitted.iloc[-1]),
            'Residual': float(residual.iloc[-1]),
            'Z-Score': zscore,
            'flag': abs(zscore) > 1.25,
        }
        for pc in pc_cols:
            row[pc] = float(contribs[pc].iloc[-1])
        rows.append(row)
    return pd.DataFrame(rows)

snap = current_snapshot(decomp, df_spreads, pca, SD_WINDOW)
pc_cols = pca['loadings'].columns.tolist()
pc_labels = {'PC1': 'PC1 (Reg)', 'PC2': 'PC2 (Diff)'}

latest = decomp['scores_full'].index[-1].date()
print(f'=== Snapshot | {MODE}: {SELECTED} | {latest} ===')
print()

pc_hdr_parts = '  '.join(f'{pc_labels.get(pc, pc):>11}' for pc in pc_cols)
print(f'{"Country":<12}  {"Actual":>8}  {"Fitted":>8}  {"Residual":>9}  {"Z-Score":>8}  {pc_hdr_parts}')

for _, row in snap.iterrows():
    flag = ' *' if row['flag'] else '  '
    zscore_str = f'{row["Z-Score"]:>+7.1f}{flag}'
    pc_vals = '  '.join(f'{row[pc]:>+10.1f} ' for pc in pc_cols)
    print(f'{row["Country"]:<12}  {row["Actual"]:>8.1f}  {row["Fitted"]:>8.1f}  '
          f'{row["Residual"]:>+9.1f}  {zscore_str}  {pc_vals}')


In [ ]:
# add curve tenors parameter
CURVE_TENORS = ['6Y', '7Y', '8Y', '9Y', '10Y', '11Y', '12Y', '13Y', '14Y', '15Y']

curve_results = {}
for tenor in CURVE_TENORS:
    sp = build_spread_panel(COUNTRIES, 'tenor', tenor, SECTORS, df_ust_cmt, LOOKBACK_START)
    pca_t = run_pca(sp, N_PCS)
    decomp_t = decompose(pca_t, sp)
    curve_results[tenor] = {'pca': pca_t, 'decomp': decomp_t, 'spreads': sp}
    v1 = pca_t['var_explained'][0] * 100
    v2 = pca_t['var_explained'][1] * 100 if pca_t['n_pcs'] >= 2 else 0.0
    n_obs = len(sp.dropna())
    print(f'{tenor}: {n_obs} obs, PC1 var={v1:.1f}%, PC2 var={v2:.1f}%')


In [ ]:
# --- Cell 16: Cross-Curve Loadings (FOCUS_COUNTRY) ---
pc1_loads, pc2_loads = [], []
for t in CURVE_TENORS:
    lds = curve_results[t]['pca']['loadings']
    pc1_loads.append(float(lds.loc[FOCUS_COUNTRY, 'PC1']))
    pc2_loads.append(float(lds.loc[FOCUS_COUNTRY, 'PC2']) if 'PC2' in lds.columns else 0.0)

x = np.arange(len(CURVE_TENORS))
width = 0.35
fig, ax = plt.subplots(figsize=(13, 4))
bars1 = ax.bar(x - width / 2, pc1_loads, width=width * 0.9, color=COLORS['pc1'],
               label='PC1', edgecolor='white')
bars2 = ax.bar(x + width / 2, pc2_loads, width=width * 0.9, color=COLORS['pc2'],
               label='PC2', edgecolor='white')
for bar, v in zip(bars1, pc1_loads):
    va = 'bottom' if v >= 0 else 'top'
    ax.text(bar.get_x() + bar.get_width() / 2, v + (0.005 if v >= 0 else -0.005),
            f'{v:.2f}', ha='center', va=va, fontsize=7)
for bar, v in zip(bars2, pc2_loads):
    va = 'bottom' if v >= 0 else 'top'
    ax.text(bar.get_x() + bar.get_width() / 2, v + (0.005 if v >= 0 else -0.005),
            f'{v:.2f}', ha='center', va=va, fontsize=7)
ax.axhline(0, color='#555555', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(CURVE_TENORS)
ax.set_title(f'{FOCUS_COUNTRY} — PC Loadings Across Curve')
ax.set_ylabel('Loading')
ax.legend()
plt.tight_layout()
plt.show()

# table with |PC1| rank
ranks = pd.Series([abs(v) for v in pc1_loads]).rank(ascending=False).astype(int).tolist()
print(f'\n=== {FOCUS_COUNTRY} Loadings Across Curve ===\n')
print(f'{"Tenor":>7}  {"PC1 Loading":>12}  {"PC2 Loading":>12}  {"|PC1| Rank":>10}')
for t, l1, l2, r in zip(CURVE_TENORS, pc1_loads, pc2_loads, ranks):
    print(f'{t:>7}  {l1:>12.4f}  {l2:>12.4f}  {r:>10}')


In [ ]:
# --- Cell 17: Cross-Curve PC Scores (FOCUS_COUNTRY) ---
import matplotlib.cm as cm

n_t = len(CURVE_TENORS)
blue_colors = [cm.Blues(0.3 + 0.7 * i / (n_t - 1)) for i in range(n_t)]
orange_colors = [cm.Oranges(0.3 + 0.7 * i / (n_t - 1)) for i in range(n_t)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8))
fig.suptitle(f'PC Scores Across Curve — {FOCUS_COUNTRY} ({LOOKBACK_START} onwards)',
             fontsize=12, fontweight='bold')

for i, tenor in enumerate(CURVE_TENORS):
    s = curve_results[tenor]['decomp']['scores_full']
    ax1.plot(s.index, s['PC1'], color=blue_colors[i], linewidth=0.9, label=tenor, alpha=0.9)
    if 'PC2' in s.columns:
        ax2.plot(s.index, s['PC2'], color=orange_colors[i], linewidth=0.9, label=tenor, alpha=0.9)

for ax, title in [(ax1, 'PC1 — Regional Level'), (ax2, 'PC2 — Differentiation')]:
    ax.axhline(0, color='#999999', linewidth=0.7)
    ax.set_ylabel('Score (bps)')
    ax.set_title(title)
    ax.legend(ncol=n_t, loc='upper left', fontsize=7)

ax2.set_xlabel('Date')
plt.tight_layout()
plt.show()


In [ ]:
# --- Cell 18: Cross-Curve Decomposition Grid (FOCUS_COUNTRY) ---
nrows, ncols = 2, 5
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 7), sharex=True, sharey=True)
axes_flat = axes.flatten()

# global y range for shared axis
all_vals = []
for tenor in CURVE_TENORS:
    all_vals.extend(curve_results[tenor]['spreads'][FOCUS_COUNTRY].dropna().values)
y_min = min(all_vals)
y_max = max(all_vals)
pad = (y_max - y_min) * 0.06
y_min -= pad
y_max += pad

pc_color_map = {'PC1': COLORS['pc1'], 'PC2': COLORS['pc2'], 'PC3': COLORS['pc3']}

for idx, tenor in enumerate(CURVE_TENORS):
    ax = axes_flat[idx]
    pca_t = curve_results[tenor]['pca']
    decomp_t = curve_results[tenor]['decomp']
    actual = curve_results[tenor]['spreads'][FOCUS_COUNTRY].dropna()
    contribs = decomp_t['contributions'][FOCUS_COUNTRY].reindex(actual.index).dropna()
    means_val = float(pca_t['means'][FOCUS_COUNTRY])
    pc_cols = pca_t['loadings'].columns.tolist()

    base = pd.Series(means_val, index=actual.index)
    cumstack = base.copy()
    ax.fill_between(base.index, 0, base, color=COLORS['mean'], alpha=0.35)
    for pc in pc_cols:
        if pc in contribs.columns:
            top = cumstack + contribs[pc]
            ax.fill_between(contribs.index, cumstack, top,
                            color=pc_color_map.get(pc, COLORS['mean']), alpha=0.4)
            cumstack = top
    ax.plot(actual.index, actual, color=COLORS['actual'], linewidth=1.2, zorder=5)
    ax.set_title(tenor, fontsize=9, pad=3)
    ax.set_ylim(y_min, y_max)

    row_i, col_i = idx // ncols, idx % ncols
    if col_i != 0:
        ax.tick_params(labelleft=False)
    if row_i != nrows - 1:
        ax.tick_params(labelbottom=False)
    else:
        ax.tick_params(axis='x', labelrotation=30, labelsize=7)

fig.suptitle(f'{FOCUS_COUNTRY} — Spread Decomposition Across Curve',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# --- Cell 19: Cross-Curve Residual Heatmap ---
import matplotlib.colors as mcolors

def cross_curve_scan(curve_results, focus_country, curve_tenors, sd_window):
    rows = []
    for tenor in curve_tenors:
        decomp_t = curve_results[tenor]['decomp']
        actual = curve_results[tenor]['spreads'][focus_country].dropna()
        residual = decomp_t['residuals'][focus_country].reindex(actual.index).dropna()
        contribs = decomp_t['contributions'][focus_country].reindex(actual.index).dropna()
        roll_sd = residual.rolling(sd_window).std()
        resid_val = float(residual.iloc[-1])
        sd_val = float(roll_sd.iloc[-1])
        zscore = resid_val / sd_val if sd_val > 0 else 0.0
        pc1_20d = (float(contribs['PC1'].iloc[-1] - contribs['PC1'].iloc[-21])
                   if 'PC1' in contribs.columns and len(contribs) >= 21 else float('nan'))
        pc2_20d = (float(contribs['PC2'].iloc[-1] - contribs['PC2'].iloc[-21])
                   if 'PC2' in contribs.columns and len(contribs) >= 21 else float('nan'))
        rows.append({'Tenor': tenor, 'Residual': resid_val, 'Z-Score': zscore,
                     'PC1 20d Chg': pc1_20d, 'PC2 20d Chg': pc2_20d})
    return pd.DataFrame(rows).set_index('Tenor')

scan_df = cross_curve_scan(curve_results, FOCUS_COUNTRY, CURVE_TENORS, SD_WINDOW)
latest_scan_date = curve_results[CURVE_TENORS[0]]['spreads'].index[-1].date()

# print table
print(f'=== {FOCUS_COUNTRY} Cross-Curve Scan | {latest_scan_date} ===\n')
print(f'{"Tenor":>7}  {"Residual":>10}  {"Z-Score":>9}  {"PC1 20d Chg":>13}  {"PC2 20d Chg":>13}')
for tenor, row in scan_df.iterrows():
    flag = '*' if abs(row['Z-Score']) > 1.25 else ' '
    print(f'{tenor:>7}  {row["Residual"]:>+9.1f}   {row["Z-Score"]:>+7.1f}{flag}  '
          f'{row["PC1 20d Chg"]:>+12.1f}   {row["PC2 20d Chg"]:>+12.1f}')

# matplotlib colored table
zmax = max(abs(scan_df['Z-Score'].max()), abs(scan_df['Z-Score'].min()), 2.0)
col_labels = ['Residual (bps)', 'Z-Score', 'PC1 20d Chg', 'PC2 20d Chg']
table_data, cell_colors = [], []
for tenor, row in scan_df.iterrows():
    z = row['Z-Score']
    norm = min(abs(z) / zmax * 0.7 + 0.15, 1.0)
    zcolor = (mcolors.to_hex(plt.cm.Reds(norm)) if z > 0
              else mcolors.to_hex(plt.cm.Blues(norm)))
    flag = '*' if abs(z) > 1.25 else ''
    row_bg = '#FFF5F5' if abs(z) > 1.25 else 'white'
    table_data.append([f'{row["Residual"]:+.1f}', f'{z:+.1f}{flag}',
                       f'{row["PC1 20d Chg"]:+.1f}', f'{row["PC2 20d Chg"]:+.1f}'])
    cell_colors.append([row_bg, zcolor, row_bg, row_bg])

fig, ax = plt.subplots(figsize=(9, len(CURVE_TENORS) * 0.5 + 1.5))
ax.set_axis_off()
tbl = ax.table(cellText=table_data, rowLabels=list(scan_df.index),
               colLabels=col_labels, cellColours=cell_colors,
               loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.15, 1.5)
ax.set_title(f'{FOCUS_COUNTRY} — Cross-Curve Residual Scan | {latest_scan_date}',
             fontsize=11, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


In [ ]:
# --- Cell 20: Curve Trade Scanner ---
from itertools import combinations

zscores_ccs = {t: float(scan_df.loc[t, 'Z-Score']) for t in CURVE_TENORS}
actuals_ccs = {t: float(curve_results[t]['spreads'][FOCUS_COUNTRY].dropna().iloc[-1])
               for t in CURVE_TENORS}
latest_ccs = curve_results[CURVE_TENORS[0]]['spreads'].index[-1].date()

trades = []
for ti, tj in combinations(CURVE_TENORS, 2):
    zi, zj = zscores_ccs[ti], zscores_ccs[tj]
    zdiff = zi - zj
    if abs(zdiff) > 1.5:
        spread_diff = actuals_ccs[ti] - actuals_ccs[tj]
        direction = (f'Flattener (buy {ti}, sell {tj})' if zdiff > 0
                     else f'Steepener (sell {ti}, buy {tj})')
        trades.append({'Pair': f'{ti} vs {tj}', 'Z(i)': zi, 'Z(j)': zj,
                       'Z-Diff': zdiff, 'Direction': direction, 'Spread Diff': spread_diff})

trades.sort(key=lambda x: abs(x['Z-Diff']), reverse=True)

print(f'=== Curve Trade Scanner | {FOCUS_COUNTRY} | {latest_ccs} ===\n')
if trades:
    print(f'{"Pair":<14}  {"Z(short)":>9}  {"Z(long)":>8}  {"Z-Diff":>7}  '
          f'{"Direction":<44}  {"Spread Diff":>11}')
    for t in trades:
        print(f'{t["Pair"]:<14}  {t["Z(i)"]:>+9.2f}  {t["Z(j)"]:>+8.2f}  '
              f'{t["Z-Diff"]:>+7.2f}  {t["Direction"]:<44}  {t["Spread Diff"]:>+10.1f} bps')
else:
    print('No qualifying pairs found.')


In [ ]:
# --- Cell 21: Positioning Overlay ---
def classify_cell(z, mom):
    row = 'Cheap' if z > 1.25 else ('Rich' if z < -1.25 else 'Neutral')
    col = 'Compressive' if mom > 10 else ('Widening' if mom < -10 else 'Neutral')
    return row, col

cell_map = {
    ('Cheap', 'Compressive'): 'A', ('Cheap', 'Neutral'): 'B', ('Cheap', 'Widening'): 'C',
    ('Neutral', 'Compressive'): 'D', ('Neutral', 'Neutral'): 'E', ('Neutral', 'Widening'): 'F',
    ('Rich', 'Compressive'): 'G', ('Rich', 'Neutral'): 'H', ('Rich', 'Widening'): 'I',
}

# step 1: current readings
actual_fc = df_spreads[FOCUS_COUNTRY].dropna()
resid_fc = decomp['residuals'][FOCUS_COUNTRY].reindex(actual_fc.index).dropna()
roll_sd_fc = resid_fc.rolling(SD_WINDOW).std()
curr_z = (float(resid_fc.iloc[-1] / roll_sd_fc.iloc[-1])
          if float(roll_sd_fc.iloc[-1]) > 0 else 0.0)
curr_mom = float(roll_factor[20]['PC1'].iloc[-1])
pc1_sc = decomp['scores_full']['PC1']
curr_pc1_pctile = float((pc1_sc < pc1_sc.iloc[-1]).mean() * 100)
curr_date_pos = actual_fc.index[-1].date()

curr_row_lbl, curr_col_lbl = classify_cell(curr_z, curr_mom)
curr_cell = cell_map[(curr_row_lbl, curr_col_lbl)]

# step 3: print matrix
print(f'=== Positioning Framework | {FOCUS_COUNTRY} | {MODE}: {SELECTED} | {curr_date_pos} ===')
print()
print(f'Current: Residual z = {curr_z:+.1f} ({curr_row_lbl}) | PC1 20d mom = {curr_mom:+.1f} bps ({curr_col_lbl})')
direction_str = 'above' if curr_pc1_pctile >= 50 else 'below'
print(f'PC1 level percentile: {curr_pc1_pctile:.0f}th ({direction_str} median)')
print()

rows_order = ['Cheap', 'Neutral', 'Rich']
cols_order = ['Compressive', 'Neutral', 'Widening']
row_desc = {'Cheap': '(z>+1.25)', 'Neutral': '(-1.25 to +1.25)', 'Rich': '(z<-1.25)'}
col_desc = {'Compressive': '(mom>+10)', 'Neutral': '(-10 to +10)', 'Widening': '(mom<-10)'}

print(f'{"":28}  {"Compressive":>14}  {"Neutral":>14}  {"Widening":>14}')
print(f'{"":28}  {col_desc["Compressive"]:>14}  {col_desc["Neutral"]:>14}  {col_desc["Widening"]:>14}')
for r in rows_order:
    row_hdr = f'{r} {row_desc[r]}'
    cells_str = ''
    for c in cols_order:
        lbl = cell_map[(r, c)]
        marker = ' <--' if (r == curr_row_lbl and c == curr_col_lbl) else '    '
        cells_str += f'  [{lbl}]{marker:5}'
    print(f'{row_hdr:<28}{cells_str}')

# step 4: historical performance
zscore_ts = (resid_fc / roll_sd_fc).dropna()
mom_ts = roll_factor[20]['PC1'].dropna()
common_idx = zscore_ts.index.intersection(mom_ts.index)
zscore_ts = zscore_ts.loc[common_idx]
mom_ts = mom_ts.loc[common_idx]

cell_ts = pd.Series(
    [cell_map[classify_cell(float(zscore_ts.loc[d]), float(mom_ts.loc[d]))]
     for d in common_idx],
    index=common_idx
)
fwd_20 = actual_fc.diff(20).shift(-20).reindex(common_idx)
fwd_40 = actual_fc.diff(40).shift(-40).reindex(common_idx)

hit_fn = {
    'A': lambda x: x < 0, 'B': lambda x: x < 0, 'C': lambda x: x < 0,
    'D': lambda x: x < 0, 'E': lambda x: True,   'F': lambda x: x > 0,
    'G': lambda x: x > 0, 'H': lambda x: x > 0,  'I': lambda x: x > 0,
}

print()
print(f'{"Cell":>6}  {"Days":>6}  {"Avg 20d Chg":>13}  {"Avg 40d Chg":>13}  {"Hit Rate 20d":>12}')
for lbl in ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I']:
    mask = cell_ts == lbl
    n_days = int(mask.sum())
    f20 = fwd_20[mask].dropna()
    f40 = fwd_40[mask].dropna()
    avg20 = float(f20.mean()) if len(f20) else float('nan')
    avg40 = float(f40.mean()) if len(f40) else float('nan')
    hits = [hit_fn[lbl](v) for v in f20 if not pd.isna(v)]
    hit_rate = sum(hits) / len(hits) * 100 if hits else float('nan')
    a20 = f'{avg20:>+11.1f} bps' if not pd.isna(avg20) else '            N/A'
    a40 = f'{avg40:>+11.1f} bps' if not pd.isna(avg40) else '            N/A'
    hr  = f'{hit_rate:>11.0f}%'  if not pd.isna(hit_rate) else '            N/A'
    cur = ' <--' if lbl == curr_cell else ''
    print(f'  [{lbl}]  {n_days:>6}  {a20}  {a40}  {hr}{cur}')

# step 5: footnote
n_total = len(common_idx)
last_date_pos = common_idx[-1].date()
print()
print('---')
print('Threshold definitions:')
print(f'- Residual z-score: computed as residual / {SD_WINDOW}d rolling SD. |z| > 1.25 = dislocation zone.')
print('- PC1 momentum: 20-day rolling sum of PC1 factor returns. |mom| > 10 bps = directional regime.')
print('- Forward changes measure actual spread change (not residual) over the horizon.')
print('- Hit rate: % of days where spread moved in the mean-reverting direction for dislocated cells,')
print('  or in the momentum direction for neutral-residual cells.')
print(f'- Historical sample: {LOOKBACK_START} to {last_date_pos}, {n_total} business days.')
